# MoCM mid-scale verification (Colab)

Runs the decisive mid-scale battery that does not fit the laptop's time budget:
distill-easy-layers + MoCM frozen probes + domain eval on Qwen3.5-2B, and
frozen probes on Qwen3.5-9B. All outputs go to Drive (sessions are ephemeral).

Runtime: GPU (T4 or better), high-RAM if offered. Budget: ~2-3 h total.
Requires: Hugging Face account (Qwen license acceptance) via `HF_TOKEN` secret.

In [ ]:
!pip install -q transformers accelerate sentencepiece huggingface_hub datasets pytest 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv | head -3
!free -g | head -2

In [ ]:
import os
os.environ['OUT'] = '/content/drive/MyDrive/moeme_colab'
!mkdir -p $OUT
# attach repo: upload moeme-repo.tar.gz to Drive once (see colab/README.md),
# or clone if public. Tarball path below:
!tar -xzf /content/drive/MyDrive/moeme-repo.tar.gz -C /content 2>&1 | tail -1 || echo 'need tarball'

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!cd /content/moeme && python3 -c "from huggingface_hub import login; login()" 2>&1 | tail -1

In [ ]:
# 1. 2B dense baselines + depth contribution map (decides easy layers)
!cd /content/moeme && python3 experiments/mocm_depth.py --model Qwen/Qwen3.5-2B --out $OUT/depth_2B.json 2>&1 | tail -8

In [ ]:
# 2. distill easy-4 at 25% (the 1.36x laptop result — reproduce at scale)
# replace LAYERS with the bottom-4 from cell above
!cd /content/moeme && python3 experiments/mocm_distill.py --model Qwen/Qwen3.5-2B --layers 1 0 10 9 --frac 0.25 --steps 1500 --save $OUT/distill_2B.pt --out $OUT/distill_2B.json 2>&1 | tail -4

In [ ]:
# 3. frozen MoCM oracle on 9B (no training: range of widths, real weights)
# 9B bf16 needs ~19GB: runs layer-streamed on CPU/RAM, slow but fits high-RAM runtimes
!cd /content/moeme && python3 experiments/mocm_poc.py --model Qwen/Qwen3.5-9B --frozen-only --cell both --out $OUT/mocm_9B_frozen.json 2>&1 | tail -8

In [ ]:
# 4. behavior battery on the distilled 2B (tasks + text + speed)
!cd /content/moeme && python3 experiments/mocm_distill_eval.py --model Qwen/Qwen3.5-2B --ckpt $OUT/distill_2B.pt --thresh 0.40 --out $OUT/eval_2B.json 2>&1 | tail -5
!cp $OUT/*.json /content/drive/MyDrive/moeme_colab/ 2>/dev/null; ls -la $OUT/